# Introduction partition example

Compare GEB, an equal-width histogram, and an oracle contour histogram on simulated two-dimensional data. Run this notebook to save the predictive scores, timings, and partition figure. No saved train/test splits are required.


In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import pandas as pd
from scipy.stats import qmc

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'methods').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'methods').exists():
    raise FileNotFoundError('Open this notebook from the GEB_experiments project.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from methods.GEB import GEBTree

In [ ]:
RANDOM_SEED = 2002
N_REPETITIONS = 1
N_TRAIN = 100_000
N_TEST = 100_000

GEB_ALPHA = 1.0
GEB_USE_GRID = True  # True: regular M-grid; False: observed-data split search
GEB_M = 32  # Number of grid subintervals
EPS = 1e-300

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'introduction_example'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = OUTPUT_DIR / 'results.csv'
SUMMARY_FILE = OUTPUT_DIR / 'summary.csv'
FIGURE_FILE = OUTPUT_DIR / 'intro_partition.pdf'

def sample_marginal(shape, rng):
    use_beta = rng.random(shape) < 0.5
    values = rng.random(shape)
    values[use_beta] = rng.beta(2.0, 2.0, size=use_beta.sum())
    return values

def sample_joint(n, rng):
    return sample_marginal((n, 2), rng)

def marginal_density(x):
    x = np.asarray(x)
    inside = (x >= 0.0) & (x <= 1.0)
    return np.where(inside, 0.5 + 3.0 * x * (1.0 - x), 0.0)

def true_density(points):
    points = np.asarray(points)
    return marginal_density(points[:, 0]) * marginal_density(points[:, 1])

if GEB_ALPHA <= 0:
    raise ValueError('GEB_ALPHA must be positive.')
if not isinstance(GEB_USE_GRID, (bool, np.bool_)):
    raise TypeError('GEB_USE_GRID must be True or False.')
if GEB_USE_GRID and GEB_M < 2:
    raise ValueError('GEB_M must be at least 2.')
split_search = f'm={GEB_M} grid' if GEB_USE_GRID else 'observed data'
print(f'GEB: alpha={GEB_ALPHA:g}, split search={split_search}')
print(f'Output: {OUTPUT_DIR}')

## Estimators

For the contour partition, deterministic Sobol points approximate Lebesgue area. Thresholds are quantiles of $p^*(x)$ under the uniform distribution on $[0,1]^2$; consequently each contour band has approximately equal area.

In [ ]:
# Fixed deterministic integration design, independent of every training sample.
calibration_points = qmc.Sobol(d=2, scramble=False).random_base2(m=18)
calibration_density = true_density(calibration_points)

def fit_geb(train):
    tree = GEBTree(
        dimension_x=2, lower_global=np.zeros(2), upper_global=np.ones(2),
        alpha=GEB_ALPHA, dataset=train, grid=GEB_USE_GRID,
        m=GEB_M,
    )
    tree.tree_joint_density_set(tree.root)
    return tree

def predict_geb(tree, points):
    return tree.tree_joint_density_batch(points)

def fit_equal_width(train, target_cells):
    bins_per_axis = max(1, int(round(np.sqrt(target_cells))))
    edges = np.linspace(0.0, 1.0, bins_per_axis + 1)
    counts, _ = np.histogramdd(train, bins=(edges, edges))
    density = counts * bins_per_axis**2 / len(train)
    return edges, density

def predict_equal_width(model, points):
    edges, density = model
    k = len(edges) - 1
    index = np.clip(np.searchsorted(edges, points, side='right') - 1, 0, k - 1)
    return density[index[:, 0], index[:, 1]]

def contour_band_index(density_values, thresholds):
    return np.searchsorted(thresholds[1:-1], density_values, side='right')

def fit_oracle_contour(train, n_bands):
    probabilities = np.linspace(0.0, 1.0, n_bands + 1)
    thresholds = np.quantile(calibration_density, probabilities)
    thresholds[0], thresholds[-1] = -np.inf, np.inf

    calibration_index = contour_band_index(calibration_density, thresholds)
    areas = np.bincount(calibration_index, minlength=n_bands) / len(calibration_index)
    train_index = contour_band_index(true_density(train), thresholds)
    counts = np.bincount(train_index, minlength=n_bands)
    heights = counts / (len(train) * areas)
    return thresholds, areas, heights

def predict_oracle_contour(model, points):
    thresholds, _, heights = model
    index = contour_band_index(true_density(points), thresholds)
    return heights[index]

## Run the experiment

Timing includes model fitting and evaluation on the test observations, but excludes data generation. Before the recorded GEB fit, one complete untimed fit on the same training sample warms the JAX compilation cache; consequently the reported GEB time excludes compilation. The true-density timing contains only its evaluation on the test observations.

In [ ]:
COLUMNS = [
    'repetition', 'method', 'test_log_score', 'running_time',
    'n_cells', 'oracle_log_score',
]
results = pd.DataFrame(columns=COLUMNS)
for repetition in range(1, N_REPETITIONS + 1):
    rng = np.random.default_rng(RANDOM_SEED + repetition - 1)
    train = sample_joint(N_TRAIN, rng)
    test = sample_joint(N_TEST, rng)

    # Warm up JAX on the training sample before timing.
    _warmup_tree = fit_geb(train)
    del _warmup_tree
    start = time.perf_counter()
    oracle_prediction = true_density(test)
    oracle_score = float(np.log(oracle_prediction).mean())
    oracle_time = time.perf_counter() - start

    start = time.perf_counter()
    tree = fit_geb(train)
    geb_prediction = predict_geb(tree, test)
    geb_time = time.perf_counter() - start
    n_leaves = len(tree.collect_leaves()[0])

    start = time.perf_counter()
    equal_width = fit_equal_width(train, n_leaves)
    equal_prediction = predict_equal_width(equal_width, test)
    equal_time = time.perf_counter() - start
    equal_cells = equal_width[1].size

    start = time.perf_counter()
    contour_bands = max(1, int(0.5*n_leaves ** (2.0 / 3.0)))
    contour = fit_oracle_contour(train, contour_bands)
    contour_prediction = predict_oracle_contour(contour, test)
    contour_time = time.perf_counter() - start

    rows = [
        (repetition, 'Equal-width histogram', np.log(np.maximum(equal_prediction, EPS)).mean(), equal_time, equal_cells, oracle_score),
        (repetition, 'Oracle contour histogram', np.log(np.maximum(contour_prediction, EPS)).mean(), contour_time, contour_bands, oracle_score),
        (repetition, 'GEB', np.log(np.maximum(geb_prediction, EPS)).mean(), geb_time, n_leaves, oracle_score),
        (repetition, 'True density', oracle_score, oracle_time, 0, oracle_score),
    ]
    results = pd.concat([results, pd.DataFrame(rows, columns=COLUMNS)], ignore_index=True)
    results = results.sort_values(['repetition', 'method'])
    temporary = RESULTS_FILE.with_suffix('.csv.tmp')
    results.to_csv(temporary, index=False)
    temporary.replace(RESULTS_FILE)
    print(
        f'{repetition:02d}: leaves={n_leaves}, contour bands={contour_bands}, '
        f'equal={rows[0][2]:.4f}, contour={rows[1][2]:.4f}, '
        f'GEB={rows[2][2]:.4f}, truth={rows[3][2]:.4f}'
    )

print(f'Saved {len(results)} rows to {RESULTS_FILE}')

## Predictive scores and timings

Report the test log score and elapsed time for this single simulation. GEB timing follows the compilation warm-up described above.


In [ ]:
summary = (
    results.set_index('method')[['test_log_score', 'running_time', 'n_cells']]
    .rename(columns={
        'test_log_score': 'predictive_log_score',
        'running_time': 'time_seconds',
    })
)
summary.to_csv(SUMMARY_FILE)
display(summary.round(6))

## Partition figure

The heat maps use the fitted models from the final repetition.

In [ ]:
grid_size = 500
coordinates = np.linspace(0.0, 1.0, grid_size)
x1, x2 = np.meshgrid(coordinates, coordinates, indexing='xy')
grid_points = np.column_stack([x1.ravel(), x2.ravel()])
surfaces = [
    true_density(grid_points).reshape(grid_size, grid_size),
    predict_equal_width(equal_width, grid_points).reshape(grid_size, grid_size),
    predict_oracle_contour(contour, grid_points).reshape(grid_size, grid_size),
    predict_geb(tree, grid_points).reshape(grid_size, grid_size),
]
titles = [
    '(a) True density', '(b) Equal-width',
    '(c) Contour-level', '(d) GEB tree',
]

normalization = Normalize(vmin=0.0, vmax=max(surface.max() for surface in surfaces))
fig, axes = plt.subplots(1, 4, figsize=(16.0, 4.1), constrained_layout=True)
for ax, surface, title in zip(axes, surfaces, titles):
    image = ax.imshow(
        surface, origin='lower', extent=(0, 1, 0, 1),
        cmap='Blues', norm=normalization, aspect='equal',
        interpolation='nearest',
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(r'$x_1$', fontsize=12)
    ax.set_ylabel(r'$x_2$', fontsize=12)
    ax.tick_params(labelsize=10)
colorbar = fig.colorbar(image, ax=axes, fraction=0.025, pad=0.02)
colorbar.set_label('Density', fontsize=12)
colorbar.ax.tick_params(labelsize=10)
fig.savefig(FIGURE_FILE, bbox_inches='tight')
plt.show()
print(f'Saved figure to {FIGURE_FILE}')